In [14]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY est introuvable dans le fichier .env")

client = OpenAI(api_key=api_key)

print("Connexion initialisée !")

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Bonjour, pouvez-vous m'aider à comprendre le fonctionnement de l'API OpenAI ?"
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

response=response.choices[0].message.content

Connexion initialisée !


## CREATION d'un systeme RAG (Prerocessing des documents/chunking/Creation de la vase de données vectorielles)

In [ ]:
from hashlib import sha256
import pickle
from docling.document_converter import DocumentConverter
from langchain_text_splitters import MarkdownHeaderTextSplitter

class Preprocessing:
    def __init__(self,size_max=8000):
        self.cache_path = "/Users/sofianeagounikaci/Agentic-end-to-end-mlops/workflows-agentic-mlops/data/cache"
        self.types = ['.pdf', '.docx', '.txt']
        self.headers = [("#", "Header 1"), ("##", "Header 2")]
        self.size_max=size_max
    def _Validate_files(self, files):
        taille=sum(os.path.getsize(f) for f in files)
        if taille>self.size_max:
            raise ValueError(f"Les fichiers dépassent la taille maximale autorisée de {self.size_max} octets.")
        for f in files:
            if not any(f.endswith(ext) for ext in self.types):
                raise ValueError(f"Le fichier {f} n'est pas d'un type autorisé. Types autorisés : {self.types}")
        return True
    def get_all_chunks(self, files):
        self._Validate_files(files)
        chunks=[]
        chunks_hash=[]
        for f in files:
            with open(f, 'rb') as file:
                hash = sha256(file.read()).hexdigest()
            cache_file = os.path.join(self.cache_path, f"{hash}.pkl")
            if os.path.exists(cache_file):
                print("the file is already cached")
                chunks.append(self.load_from_cache(cache_file))


            else:
                chunk = self.get_chunks(f)
                self.save_to_cache(cache_file, chunk)
                chunks.append(chunk)
                chunks_hash.append(sha256(str(chunk).encode("utf-8")).hexdigest())

        return chunks

    def load_from_cache(self, cache_file):
        with open(cache_file, "rb") as f:
            data = pickle.load(f)
        return data
    def save_to_cache(self, cache_file, chunks):
        with open(cache_file, "wb") as f:
            pickle.dump(chunks, f)
    def get_chunks(self, file:str) :
        """Original processing logic with Docling"""
        if  file.endswith(('.pdf', '.docx', '.txt', '.md')):

            converter = DocumentConverter()
            markdown = converter.convert(file).document.export_to_markdown()
            splitter = MarkdownHeaderTextSplitter(self.headers)
            return splitter.split_text(markdown)
        



test=Preprocessing(size_max=8000000)
type(test.get_all_chunks(["/Users/sofianeagounikaci/Agentic-end-to-end-mlops/workflows-agentic-mlops/data/external/politique_incidents_tic_scan.pdf","/Users/sofianeagounikaci/Agentic-end-to-end-mlops/workflows-agentic-mlops/data/external/politique_incidents_tic_scan.pdf"]))

the file is already cached
the file is already cached
[[Document(metadata={'Header 2': 'Politique de gestion des incidents liés aux TIC'}, page_content="Détection, classification, traitement et notification des incidents  \n| Référence       | POL-SEC-014                                                |\n|-----------------|------------------------------------------------------------|\n| Version         | 2.3                                                        |\n| Date d'effet    | 1er mars 2026                                              |\n| Propriétaire    | Direction de la Sécurité des Systèmes d'Information (DSSI) |\n| Approbation     | Comité des risques opérationnels, 12 février 2026          |\n| Prochaine revue | Février 2027                                               |\n| Classification  | Interne                                                    |  \nAe  t   e       t e : uti d'analyse documentaire. Il ne constitue ni un modèle de politique ni un avis juridique."), D

In [ ]:
import hashlib
import json
import logging

from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings


logger = logging.getLogger(__name__)

COLLECTION_NAME = "documents"
EMBEDDING_MODEL = "text-embedding-3-small"


class RetrieverBuilder:
    """Manage the Chroma database and build the hybrid retriever.

    Parameters
    ----------
    embeddings : langchain_core.embeddings.Embeddings, optional
        Embedding model. Defaults to OpenAI ``text-embedding-3-small``.
    persist_directory : str, optional
        Database directory. Defaults to ``settings.CHROMA_DB_PATH``.
    collection_name : str, default=COLLECTION_NAME
        Name of the Chroma collection.
    """

    def __init__(
        self,
        persist_directory="./chroma_db",
        collection_name=COLLECTION_NAME,
    ):
        self.embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
        self.persist_directory = persist_directory 
        self.collection_name = collection_name
        self.vector_store = None

    def create_vector_store(self):
        """Create the Chroma database, or open it if it already exists.

        Returns
        -------
        langchain_chroma.Chroma
            The persistent vector store.
        """
        self.vector_store = Chroma(
            collection_name=self.collection_name,
            embedding_function=self.embeddings,
            persist_directory=self.persist_directory,
            collection_configuration={"hnsw": {"space": "cosine"}},
        )
        logger.info("Vector store '%s' is ready.", self.collection_name)
        return self.vector_store

    def add_documents(self, documents):
        """Add documents to the database and skip those already stored.

        Parameters
        ----------
        documents : list of langchain_core.documents.Document
            Documents to add.

        Returns
        -------
        int
            Number of documents actually added.
        """
        vector_store = self._get_vector_store()
        cleaned = filter_complex_metadata(
            [doc for doc in documents if doc.page_content.strip()]
        )
        unique = {self._make_document_id(doc): doc for doc in cleaned}
        if not unique:
            return 0

        existing = set(vector_store.get(ids=list(unique), include=[])["ids"])
        new_documents = {
            doc_id: doc for doc_id, doc in unique.items() if doc_id not in existing
        }
        if new_documents:
            vector_store.add_documents(
                documents=list(new_documents.values()),
                ids=list(new_documents),
            )
        logger.info(
            "Added %d documents, skipped %d.",
            len(new_documents),
            len(documents) - len(new_documents),
        )
        return len(new_documents)

    def build_hybrid_retriever(self, k=None, weights=None):
        """Build a hybrid BM25 and vector retriever from the database.

        The BM25 index lives in memory and reflects the database at call time.
        Call this method again after adding documents.

        Parameters
        ----------
        k : int, optional
            Results per retriever. Defaults to ``settings.VECTOR_SEARCH_K``.
        weights : list of float, optional
            BM25 and vector weights. Defaults to
            ``settings.HYBRID_RETRIEVER_WEIGHTS``.

        Returns
        -------
        langchain_classic.retrievers.EnsembleRetriever
            The hybrid retriever.

        Raises
        ------
        ValueError
            If the database is empty.
        """
        vector_store = self._get_vector_store()
        documents = self._load_documents()
        if not documents:
            raise ValueError("Cannot build a retriever from an empty database.")

        k = k or settings.VECTOR_SEARCH_K
        bm25_retriever = BM25Retriever.from_documents(documents, k=k)
        vector_retriever = vector_store.as_retriever(search_kwargs={"k": k})
        logger.info("Hybrid retriever built on %d documents.", len(documents))
        return EnsembleRetriever(
            retrievers=[bm25_retriever, vector_retriever],
            weights=weights or settings.HYBRID_RETRIEVER_WEIGHTS,
        )

    def _get_vector_store(self):
        """Return the vector store, opening it on first use.

        Returns
        -------
        langchain_chroma.Chroma
            The persistent vector store.
        """
        if self.vector_store is None:
            return self.create_vector_store()
        return self.vector_store

    def _load_documents(self):
        """Read every document stored in the database.

        Returns
        -------
        list of langchain_core.documents.Document
            Stored documents.
        """
        stored = self._get_vector_store().get(include=["documents", "metadatas"])
        return [
            Document(page_content=text, metadata=metadata or {})
            for text, metadata in zip(stored["documents"], stored["metadatas"])
        ]

    @staticmethod
    def _make_document_id(document):
        """Build a deterministic identifier from a document's content.

        Parameters
        ----------
        document : langchain_core.documents.Document
            Document to identify.

        Returns
        -------
        str
            Identifier that changes when the text or the metadata change.
        """
        payload = json.dumps(
            {"text": document.page_content, "metadata": document.metadata},
            ensure_ascii=False,
            sort_keys=True,
        )
        return hashlib.sha256(payload.encode("utf-8")).hexdigest()

In [57]:
from openai import OpenAI

client = OpenAI()

response = client.embeddings.create(
    input="Your text string goes here", model="text-embedding-3-small"
)

print(response.data[0].embedding)

[0.005115509033203125, 0.017242431640625, -0.018768310546875, -0.0185699462890625, -0.047271728515625, -0.030303955078125, 0.027679443359375, 0.0036468505859375, 0.01125335693359375, 0.006404876708984375, -0.001678466796875, 0.0158233642578125, -0.001255035400390625, -0.007843017578125, 0.0599365234375, 0.050323486328125, -0.0275421142578125, 0.00994110107421875, -0.04034423828125, 0.050018310546875, -0.00043082237243652344, 0.030181884765625, -0.0137176513671875, 0.032958984375, 0.0173187255859375, 0.0167999267578125, -0.0017566680908203125, 0.0204620361328125, 0.040802001953125, -0.0377197265625, -0.026153564453125, -0.050048828125, 0.024200439453125, -0.05523681640625, -0.0322265625, 0.0423583984375, 0.064697265625, 0.01473236083984375, -0.01568603515625, -0.041351318359375, 0.02215576171875, 0.007411956787109375, 0.044952392578125, 0.0070953369140625, -0.0240936279296875, 0.05242919921875, -0.0200958251953125, -0.0322265625, 0.0162811279296875, 0.046478271484375, 0.0238494873046875